In [4]:
import os 
import requests

pdf_path = "human-nutrition-text.pdf"

if not os.path.exists(pdf_path):
    print(f"[INFO] File doesn't exist, downloading....")

    url = "https://pressbooks.oer.hawaii.edu/humannutrition2/open/download?type=pdf"
    filename = pdf_path
    response = requests.get(url)
    #check if the request was successful
    if response.status_code == 200:
        with open(filename, "wb") as file:
            file.write(response.content)
        print(f"[INFO] the file has been downloaded and saved as {filename}")
    else:
        print(f"[INFO] Failed to download the file. status code: {response.status_code}")
else:
    print(f"File {pdf_path} already exists")

File human-nutrition-text.pdf already exists


In [7]:
import fitz
from tqdm.auto import tqdm

def text_formatter(text: str) -> str:
    """ Perfroms minor formatting on text."""
    cleaned_text = text.replace("\n"," ").strip()
    return cleaned_text

def open_and_read_pdf(pdf_path: str) -> list[dict]:
    doc = fitz.open(pdf_path)
    pages_and_texts = []
    for page_number, page in tqdm(enumerate(doc)):
        text = page.get_text()
        text = text_formatter(text=text)
        pages_and_texts.append({"page_number" : page_number - 41,
                                "page_char_count" : len(text),
                                "page_word_count" : len(text.split(" ")),
                                "page_sentence_count_raw" : len(text.split(". ")),
                                "page_token_count" : len(text) /4, # we assume that 1 token = ~4 characters
                                "text" : text   
        })
    return pages_and_texts

pages_and_texts = open_and_read_pdf(pdf_path=pdf_path)
pages_and_texts[:2]

0it [00:00, ?it/s]

[{'page_number': -41,
  'page_char_count': 29,
  'page_word_count': 4,
  'page_sentence_count_raw': 1,
  'page_token_count': 7.25,
  'text': 'Human Nutrition: 2020 Edition'},
 {'page_number': -40,
  'page_char_count': 0,
  'page_word_count': 1,
  'page_sentence_count_raw': 1,
  'page_token_count': 0.0,
  'text': ''}]

In [8]:
import random

random.sample(pages_and_texts, k=3)

[{'page_number': 477,
  'page_char_count': 1484,
  'page_word_count': 283,
  'page_sentence_count_raw': 11,
  'page_token_count': 371.0,
  'text': 'us begin with the basics on how to estimate energy intake, energy  requirement, and energy output. Then we will consider the other  factors that play a role in maintaining energy balance and hence,  body weight.  Estimating Energy Requirement  To maintain body weight you have to balance the calories obtained  from food and beverages with the calories expended every day.  Here, we will discuss how to calculate your energy needs in  kilocalories per day so that you can determine whether your caloric  intake falls short, meets, or exceeds your energy needs. The  Institute of Medicine has devised a formula for calculating your  Estimated Energy Requirement (EER). It takes into account your  age, sex, weight, height, and physical activity level (PA). The EER is  a standardized mathematical prediction of a person’s daily energy  needs in kilocalo

In [9]:
import pandas as pd

df = pd.DataFrame(pages_and_texts)
df.head()

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count,text
0,-41,29,4,1,7.25,Human Nutrition: 2020 Edition
1,-40,0,1,1,0.00,
2,-39,320,54,1,80.00,Human Nutrition: 2020 Edition UNIVERSITY OF ...
3,-38,212,32,1,53.00,Human Nutrition: 2020 Edition by University of...
4,-37,797,145,2,199.25,Contents Preface University of Hawai‘i at Mā...


In [10]:
df.describe().round(2)

,page_number,page_char_count,page_word_count,page_sentence_count_raw,page_token_count
count,1208.00,1208.00,1208.00,1208.00,1208.00
mean,562.50,1148.00,198.30,9.97,287.00
std,348.86,560.38,95.76,6.19,140.10
min,-41.00,0.00,1.00,1.00,0.00
25%,260.75,762.00,134.00,4.00,190.50
50%,562.50,1231.50,214.50,10.00,307.88
75%,864.25,1603.50,271.00,14.00,400.88
max,1166.00,2308.00,429.00,32.00,577.00


Why would we care about token count?

Token couunt is important to think about because:
1. Embedding models don't deal with infinite tokens.
2. LLMs don't deal with infinite tokens.

for example an embedding model may have been trained to embed sequences of 384 token into numerical space (sentence-transformers 'all-mpnet-base-v2')

As fro LLMs, they can't accept infinite tokens in their context window.